# Notebook 02: Stationarity Testing

This notebook determines whether the fuel-price and macroeconomic series have stable behavior over time. It applies the ADF, KPSS, and Zivot-Andrews tests at levels, then repeats the relevant tests after first and second differencing. The results guide the differencing choices used by ARIMA, ARIMAX, and VAR models. A small p-value must always be interpreted according to the null hypothesis of the specific test.

In [2]:
import pandas as pd
from statsmodels.tsa.stattools import adfuller, kpss, zivot_andrews
import warnings
warnings.filterwarnings('ignore')

In [3]:
# --- Load and prepare data ---
df = pd.read_csv("../data/dataset_final/full_data.csv")
df['date'] = pd.to_datetime(df['month'], format='%YM%m')
df.set_index('date', inplace=True)
df.index.freq = 'MS'

num_cols = df.select_dtypes(include=['float64', 'int64']).columns

In [6]:
# --- Function: ADF + KPSS + Zivot-Andrews on a series at level ---
def stationarity_tests(series, name):
    """
    Runs ADF, KPSS, and Zivot-Andrews tests on a series at level.
    ADF H0: unit root (non-stationary)
    KPSS H0: stationary
    ZA H0: unit root, allowing for one structural break
    """
    results = {"series": name}

    adf_stat, adf_p, *_ = adfuller(series, autolag="AIC")
    results["ADF_stat"] = round(adf_stat, 4)
    results["ADF_pvalue"] = round(adf_p, 4)
    results["ADF_conclusion"] = "Stationary" if adf_p < 0.05 else "Non-stationary"

    kpss_stat, kpss_p, *_ = kpss(series, regression="c", nlags="auto")
    results["KPSS_stat"] = round(kpss_stat, 4)
    results["KPSS_pvalue"] = round(kpss_p, 4)
    results["KPSS_conclusion"] = "Non-stationary" if kpss_p < 0.05 else "Stationary"

    za_stat, za_p, _, _, za_break = zivot_andrews(series, regression="c")
    results["ZA_stat"] = round(za_stat, 4)
    results["ZA_pvalue"] = round(za_p, 4)
    results["ZA_break_date"] = series.index[za_break].strftime("%Y-%m")
    results["ZA_conclusion"] = "Stationary (with break)" if za_p < 0.05 else "Non-stationary"

    return results


# --- Function: ADF + KPSS on a series after differencing ---
def stationarity_tests_diff(series, name):
    """Runs ADF and KPSS on an already-differenced series."""
    results = {"series": name}

    adf_stat, adf_p, *_ = adfuller(series, autolag="AIC")
    results["ADF_stat"] = round(adf_stat, 4)
    results["ADF_pvalue"] = round(adf_p, 4)
    results["ADF_conclusion"] = "Stationary" if adf_p < 0.05 else "Non-stationary"

    kpss_stat, kpss_p, *_ = kpss(series, regression="c", nlags="auto")
    results["KPSS_stat"] = round(kpss_stat, 4)
    results["KPSS_pvalue"] = round(kpss_p, 4)
    results["KPSS_conclusion"] = "Non-stationary" if kpss_p < 0.05 else "Stationary"

    return results

In [5]:
# --- Step 1: Test all series at level ---
level_results = [stationarity_tests(df[col], col) for col in num_cols]
level_results_df = pd.DataFrame(level_results)
print("=== Stationarity tests at LEVEL ===")
print(level_results_df.to_string(index=False))

# --- Step 2: First-difference all series and re-test ---
df_diff1 = df[num_cols].diff().dropna()
diff1_results = [stationarity_tests_diff(df_diff1[col], col) for col in num_cols]
diff1_results_df = pd.DataFrame(diff1_results)
print("\n=== Stationarity tests after FIRST DIFFERENCING (d=1) ===")
print(diff1_results_df.to_string(index=False))

# --- Step 3: Confirm second differences for the VAR system ---
# The VAR notebook uses all six variables, including LPG and CPI.
var_cols = [
    "diesel_price_(GHC/litre)",
    "petrol_price_ghs/litre",
    "lpg_ghs/kg",
    "brent_price_usd_per_barrel",
    "usd/ghs_rate",
    "consumer_price_index",
]
df_diff2 = df[var_cols].diff().diff().dropna()
var_d2_results = [stationarity_tests_diff(df_diff2[col], col) for col in var_cols]
var_d2_results_df = pd.DataFrame(var_d2_results)
print("\n=== VAR variables after SECOND DIFFERENCING (d=2) ===")
print(var_d2_results_df.to_string(index=False))

=== Stationarity tests at LEVEL ===
                    series  ADF_stat  ADF_pvalue ADF_conclusion  KPSS_stat  KPSS_pvalue KPSS_conclusion  ZA_stat  ZA_pvalue ZA_break_date           ZA_conclusion
  diesel_price_(GHC/litre)   -0.7059      0.8452 Non-stationary     1.7488         0.01  Non-stationary  -5.2228     0.0133       2022-01 Stationary (with break)
                lpg_ghs/kg   -1.3448      0.6084 Non-stationary     1.7310         0.01  Non-stationary  -5.7158     0.0018       2022-03 Stationary (with break)
    petrol_price_ghs/litre   -0.7504      0.8333 Non-stationary     1.7097         0.01  Non-stationary  -5.3486     0.0074       2022-02 Stationary (with break)
brent_price_usd_per_barrel   -2.0584      0.2616 Non-stationary     0.9747         0.01  Non-stationary  -3.4362     0.7194       2021-01          Non-stationary
              usd/ghs_rate   -1.1479      0.6956 Non-stationary     1.6635         0.01  Non-stationary  -4.9684     0.0323       2022-03 Stationary (with